# Week 4 Tutorial: LLM Benchmarking vs Week 3 ML

This tutorial is intentionally simple.

You will:
- Load Week 3 test data
- Query three LLM endpoints
- Parse outputs into structured predictions
- Compare ML vs each LLM
- Save raw and clean result CSV files

## Step 1: Imports

What this does: imports the tools needed for file paths, table operations, parsing model output, and calling the model endpoint.

Why it matters: each import supports one part of the benchmark pipeline, so your notebook stays organized and predictable.

Principle: separate concerns. Use specific libraries for specific jobs (data handling, parsing, model calls) instead of mixing everything together.

In [1]:
from pathlib import Path
import json
import re

import numpy as np
import pandas as pd

openai = __import__('openai')
OpenAI = openai.OpenAI

## Step 2: Paths and settings

What this does: sets all configurable values in one place: input files, column names, model settings, and output file locations.

Why it matters: if your team dataset or naming changes, you only edit one cell.

Principle: parameterization improves reproducibility and reduces accidental bugs from hardcoded values scattered across cells.

In [5]:
ROOT = Path('..').resolve()

WEEK3_TEST_CSV = 'week3_artifacts/test_split.csv'
WEEK3_ML_RESULTS_CSV = 'week3_artifacts/ml_predictions.csv'

ID_COL = 'sample_id'
TEXT_COL = 'text'
LABEL_COL = 'label'
ML_PRED_COL = 'prediction'

MODEL_ENDPOINTS = [
    {'label': 'qwen2.5-7b', 'model_name': 'Qwen2.5-7B', 'host': 'localhost', 'port': 8000},
    {'label': 'qwen2.5-7b-instruct', 'model_name': 'Qwen2.5-7B-Instruct', 'host': 'localhost', 'port': 8001},
    {'label': 'qwen2.5-coder-7b-instruct', 'model_name': 'Qwen2.5-Coder-7B-Instruct', 'host': 'localhost', 'port': 8002},
]
API_KEY = 'EMPTY'
ITERATION_NUMBER = 1

OUTPUT_DIR = ROOT / 'Results'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RAW_RESULTS_PATH = OUTPUT_DIR / f'llm_results_raw_{ITERATION_NUMBER}.csv'
CLEAN_RESULTS_PATH = OUTPUT_DIR / f'llm_results_clean_{ITERATION_NUMBER}.csv'

## Step 3: Load Week 3 files

What this does: loads Week 3 test data and Week 3 ML predictions, validates expected columns, then merges them on sample_id.

Why it matters: Week 4 comparison is only fair if both ML and LLM use the same held-out test examples.

Principle: benchmark validity depends on data integrity. Always validate schema and join keys before scoring.

In [6]:
test_df = pd.read_csv(WEEK3_TEST_CSV)
ml_df = pd.read_csv(WEEK3_ML_RESULTS_CSV)

needed_test = {ID_COL, TEXT_COL, LABEL_COL}
needed_ml = {ID_COL, ML_PRED_COL}
if needed_test - set(test_df.columns):
    raise ValueError(f'Missing in test_split.csv: {sorted(needed_test - set(test_df.columns))}')
if needed_ml - set(ml_df.columns):
    raise ValueError(f'Missing in ml_predictions.csv: {sorted(needed_ml - set(ml_df.columns))}')

df = test_df[[ID_COL, TEXT_COL, LABEL_COL]].merge(
    ml_df[[ID_COL, ML_PRED_COL]], on=ID_COL, how='left'
)

print(f'Loaded rows: {len(df)}')
display(df.head())

Loaded rows: 40


,sample_id,text,label,prediction
0,1,Rapidly dividing cells with high mitotic index...,high_risk,high_risk
1,2,The culture showed stable morphology and low i...,low_risk,low_risk
2,3,Mild elevation in cytokines was detected after...,moderate_risk,moderate_risk
3,4,Gene panel indicates multiple pathogenic varia...,high_risk,high_risk
4,5,Protein expression remained within baseline an...,low_risk,low_risk


## Step 4: Prompt + parser helpers

What this does: defines a strict prompt format and a parser that turns model text into structured fields (prediction, confidence, parse status).

Why it matters: LLM responses are free-form by default, but evaluation requires deterministic, machine-readable outputs.

Principle: constrain generation and validate parsing. Reliability comes from clear output contracts plus defensive checks.

In [7]:
label_values = sorted(df[LABEL_COL].astype(str).unique().tolist())
label_set = set(label_values)

def make_prompt(text, labels):
    return (
        'You are solving a classification task.\n'
        f'Allowed labels: {json.dumps(labels)}\n'
        'Return ONLY JSON with schema: '
        '{"prediction": "<label>", "confidence": <0_to_1_float>}\n'
        'No extra text.\n\n'
        f'Text:\n{text}'
    )

def parse_response(raw_text, valid_labels):
    output = {'llm_prediction': None, 'llm_confidence': np.nan, 'parse_ok': False, 'parse_error': None}

    if not isinstance(raw_text, str) or not raw_text.strip():
        output['parse_error'] = 'empty_response'
        return output

    m = re.search(r'\{.*\}', raw_text.strip(), flags=re.DOTALL)
    candidate = m.group(0) if m else raw_text.strip()

    try:
        payload = json.loads(candidate)
    except Exception:
        output['parse_error'] = 'invalid_json'
        return output

    pred = str(payload.get('prediction', '')).strip()
    if pred not in valid_labels:
        output['parse_error'] = 'invalid_label'
        return output

    conf = payload.get('confidence', np.nan)
    try:
        conf = float(conf)
    except Exception:
        conf = np.nan

    if not np.isnan(conf) and not (0 <= conf <= 1):
        conf = np.nan

    output['llm_prediction'] = pred
    output['llm_confidence'] = conf
    output['parse_ok'] = True
    return output

## Step 5: Single-row smoke test

What this does: tests one example on each model endpoint before running the full benchmark loop.

Why it matters: this catches endpoint/port issues early and confirms all models are reachable.

Principle: validate connectivity and output format for each model before large-scale evaluation.

In [8]:
SYSTEM_PROMPT = 'You are a careful classifier that follows output format instructions.'

def query_llm(prompt, endpoint_cfg, temperature=0.0, seed=0):
    client = OpenAI(
        api_key=API_KEY,
        base_url=f"http://{endpoint_cfg['host']}:{endpoint_cfg['port']}/v1"
    )
    response = client.chat.completions.create(
        model=endpoint_cfg['model_name'],
        messages=[
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': prompt},
        ],
        temperature=temperature,
        seed=seed
    )
    return response.choices[0].message.content

example = df.iloc[0]
for endpoint_cfg in MODEL_ENDPOINTS:
    raw = query_llm(make_prompt(example[TEXT_COL], label_values), endpoint_cfg, temperature=0.0, seed=0)
    parsed = parse_response(raw, label_set)
    print(f"\nModel: {endpoint_cfg['model_name']} (port {endpoint_cfg['port']})")
    print('Raw response:')
    print(raw)
    print('Parsed:')
    print(parsed)


Model: Qwen2.5-7B (port 8000)
Raw response:
{"prediction": "high_risk", "confidence": 0.95}
Parsed:
{'llm_prediction': 'high_risk', 'llm_confidence': 0.95, 'parse_ok': True, 'parse_error': None}

Model: Qwen2.5-7B-Instruct (port 8001)
Raw response:
{"prediction": "high_risk", "confidence": 0.85}
Parsed:
{'llm_prediction': 'high_risk', 'llm_confidence': 0.85, 'parse_ok': True, 'parse_error': None}

Model: Qwen2.5-Coder-7B-Instruct (port 8002)
Raw response:
{
  "prediction": "high_risk",
  "confidence": 0.95
}
Parsed:
{'llm_prediction': 'high_risk', 'llm_confidence': 0.95, 'parse_ok': True, 'parse_error': None}


## Step 6: Full benchmark run

What this does: loops over all configured model endpoints and all test rows, storing raw responses and parsed predictions.

Why it matters: this produces side-by-side model outputs from a single consistent pipeline.

Principle: hold the dataset and prompt constant while varying models for a fair model-to-model comparison.

In [9]:
rows = []
for endpoint_cfg in MODEL_ENDPOINTS:
    print(f"\nRunning benchmark for {endpoint_cfg['model_name']} on port {endpoint_cfg['port']}...")
    for i, row in df.iterrows():
        raw = query_llm(
            make_prompt(str(row[TEXT_COL]), label_values),
            endpoint_cfg,
            temperature=0.0,
            seed=0
        )
        parsed = parse_response(raw, label_set)

        rows.append({
            ID_COL: row[ID_COL],
            'model_label': endpoint_cfg['label'],
            'model_name': endpoint_cfg['model_name'],
            'endpoint_port': endpoint_cfg['port'],
            'prompt_version': 'v1_strict_json',
            'ground_truth': str(row[LABEL_COL]),
            'ml_prediction': str(row[ML_PRED_COL]),
            'raw_response': raw,
            **parsed
        })

        if (i + 1) % 10 == 0:
            print(f"  Completed {i + 1}/{len(df)}")

results_df = pd.DataFrame(rows)
results_df.to_csv(RAW_RESULTS_PATH, index=False)
print(f'Saved raw CSV: {RAW_RESULTS_PATH}')
display(results_df.head())


Running benchmark for Qwen2.5-7B on port 8000...
  Completed 10/40
  Completed 20/40
  Completed 30/40
  Completed 40/40

Running benchmark for Qwen2.5-7B-Instruct on port 8001...
  Completed 10/40
  Completed 20/40
  Completed 30/40
  Completed 40/40

Running benchmark for Qwen2.5-Coder-7B-Instruct on port 8002...
  Completed 10/40
  Completed 20/40
  Completed 30/40
  Completed 40/40
Saved raw CSV: /home/jupyter-abby/Results/llm_results_raw_1.csv


,sample_id,model_label,model_name,endpoint_port,prompt_version,ground_truth,ml_prediction,raw_response,llm_prediction,llm_confidence,parse_ok,parse_error
0,1,qwen2.5-7b,Qwen2.5-7B,8000,v1_strict_json,high_risk,high_risk,"{""prediction"": ""high_risk"", ""confidence"": 0.95}",high_risk,0.95,True,NaN
1,2,qwen2.5-7b,Qwen2.5-7B,8000,v1_strict_json,low_risk,low_risk,"{""prediction"": ""low_risk"", ""confidence"": 0.95}",low_risk,0.95,True,NaN
2,3,qwen2.5-7b,Qwen2.5-7B,8000,v1_strict_json,moderate_risk,moderate_risk,"{""prediction"": ""moderate_risk"", ""confidence"": ...",moderate_risk,0.75,True,NaN
3,4,qwen2.5-7b,Qwen2.5-7B,8000,v1_strict_json,high_risk,high_risk,"{""prediction"": ""high_risk"", ""confidence"": 0.95}",high_risk,0.95,True,NaN
4,5,qwen2.5-7b,Qwen2.5-7B,8000,v1_strict_json,low_risk,low_risk,"{""prediction"": ""low_risk"", ""confidence"": 0.95}",low_risk,0.95,True,NaN


## Step 7: Evaluate and compare

What this does: computes ML accuracy, LLM accuracy, and invalid output rate from the parsed results.

Why it matters: this gives a direct Week 3 vs Week 4 comparison on the same data split.

Principle: evaluate both quality and reliability. Accuracy without format reliability can still fail in production workflows.

In [10]:
eval_df = results_df.copy()
eval_df['llm_prediction_filled'] = eval_df['llm_prediction'].fillna('__INVALID__')

ml_accuracy = float((eval_df['ground_truth'] == eval_df['ml_prediction']).mean())

llm_summary = eval_df.groupby(['model_label', 'model_name', 'endpoint_port'], as_index=False).agg(
    llm_accuracy=('llm_prediction_filled', lambda s: float((eval_df.loc[s.index, 'ground_truth'] == s).mean())),
    invalid_output_rate=('parse_ok', lambda s: float((~s).mean())),
    rows=('model_name', 'size')
)

summary_df = pd.concat([
    pd.DataFrame([{'model_type': 'Week3_ML', 'model_label': 'week3_ml', 'model_name': 'Week3 ML Baseline', 'endpoint_port': np.nan, 'llm_accuracy': ml_accuracy, 'invalid_output_rate': 0.0, 'rows': len(eval_df)}]),
    llm_summary.assign(model_type='Week4_LLM')
] , ignore_index=True)

display(summary_df)
display(eval_df[['model_name', 'ground_truth', 'ml_prediction', 'llm_prediction', 'parse_ok', 'parse_error']].head(12))

,model_type,model_label,model_name,endpoint_port,llm_accuracy,invalid_output_rate,rows
0,Week3_ML,week3_ml,Week3 ML Baseline,NaN,0.975,0.00,120
1,Week4_LLM,qwen2.5-7b,Qwen2.5-7B,8000.0,0.925,0.05,40
2,Week4_LLM,qwen2.5-7b-instruct,Qwen2.5-7B-Instruct,8001.0,0.975,0.00,40
3,Week4_LLM,qwen2.5-coder-7b-instruct,Qwen2.5-Coder-7B-Instruct,8002.0,0.900,0.00,40


,model_name,ground_truth,ml_prediction,llm_prediction,parse_ok,parse_error
0,Qwen2.5-7B,high_risk,high_risk,high_risk,True,NaN
1,Qwen2.5-7B,low_risk,low_risk,low_risk,True,NaN
2,Qwen2.5-7B,moderate_risk,moderate_risk,moderate_risk,True,NaN
3,Qwen2.5-7B,high_risk,high_risk,high_risk,True,NaN
4,Qwen2.5-7B,low_risk,low_risk,low_risk,True,NaN
5,Qwen2.5-7B,moderate_risk,low_risk,moderate_risk,True,NaN
6,Qwen2.5-7B,high_risk,high_risk,NaN,False,invalid_json
7,Qwen2.5-7B,low_risk,low_risk,low_risk,True,NaN
8,Qwen2.5-7B,moderate_risk,moderate_risk,moderate_risk,True,NaN
9,Qwen2.5-7B,high_risk,high_risk,NaN,False,invalid_json


## Step 8: Save clean CSV and draft summary

What this does: writes a clean submission-ready CSV and generates a short comparison paragraph for reporting.

Why it matters: deliverables should be generated by code to stay consistent and repeatable.

Principle: automate outputs for reproducibility. If you rerun the notebook, you should get the same structure every time.

In [11]:
clean_cols = [
    ID_COL, 'model_label', 'model_name', 'endpoint_port', 'prompt_version', 'ground_truth', 'ml_prediction',
    'llm_prediction', 'llm_confidence', 'parse_ok', 'parse_error'
 ]
clean_df = results_df[clean_cols].copy()
clean_df.to_csv(CLEAN_RESULTS_PATH, index=False)

# Also export per-model files following competition naming style.
for model_label, part in clean_df.groupby('model_label'):
    per_model_clean = OUTPUT_DIR / f'{model_label}_results_clean_{ITERATION_NUMBER}.csv'
    per_model_raw = OUTPUT_DIR / f'{model_label}_results_raw_{ITERATION_NUMBER}.csv'
    part.to_csv(per_model_clean, index=False)
    results_df[results_df['model_label'] == model_label].to_csv(per_model_raw, index=False)
    print(f'Saved per-model clean CSV: {per_model_clean}')
    print(f'Saved per-model raw CSV: {per_model_raw}')

print(f'\nSaved combined clean CSV: {CLEAN_RESULTS_PATH}')
print(f'Saved combined raw CSV:  {RAW_RESULTS_PATH}')

print('\nComparison Summary draft (per model):')
for _, row in llm_summary.iterrows():
    print(
        f"Week 3 ML accuracy={ml_accuracy:.3f}; "
        f"{row['model_name']} (port {int(row['endpoint_port'])}) accuracy={row['llm_accuracy']:.3f}; "
        f"invalid output rate={row['invalid_output_rate']:.3f}."
    )

Saved per-model clean CSV: /home/jupyter-abby/Results/qwen2.5-7b_results_clean_1.csv
Saved per-model raw CSV: /home/jupyter-abby/Results/qwen2.5-7b_results_raw_1.csv
Saved per-model clean CSV: /home/jupyter-abby/Results/qwen2.5-7b-instruct_results_clean_1.csv
Saved per-model raw CSV: /home/jupyter-abby/Results/qwen2.5-7b-instruct_results_raw_1.csv
Saved per-model clean CSV: /home/jupyter-abby/Results/qwen2.5-coder-7b-instruct_results_clean_1.csv
Saved per-model raw CSV: /home/jupyter-abby/Results/qwen2.5-coder-7b-instruct_results_raw_1.csv

Saved combined clean CSV: /home/jupyter-abby/Results/llm_results_clean_1.csv
Saved combined raw CSV:  /home/jupyter-abby/Results/llm_results_raw_1.csv

Comparison Summary draft (per model):
Week 3 ML accuracy=0.975; Qwen2.5-7B (port 8000) accuracy=0.925; invalid output rate=0.050.
Week 3 ML accuracy=0.975; Qwen2.5-7B-Instruct (port 8001) accuracy=0.975; invalid output rate=0.000.
Week 3 ML accuracy=0.975; Qwen2.5-Coder-7B-Instruct (port 8002) accura

### TRY THIS

- Modify the prompt wording and check parse reliability
- Add few-shot examples to the prompt
- Add macro-F1 so it matches your Week 3 metrics exactly
